In [0]:
%sql
-- drop table if exists nleshin_catalog.raw_layer.objects_description purge;

drop table nleshin_catalog.stg_layer.objects_description;

In [0]:
%sql
-- Step 1 - Create stg table 

-- drop table if exists nleshin_catalog.stg_layer.objects_description;

create table if not exists nleshin_catalog.stg_layer.objects_description(
    file_name string,
    file_surrogate_key string,
    length bigint,
    parsed_content variant,
    src_inserted_stamp timestamp,
    operation_flag string,
    hash_diff string
);

In [0]:
# Step 2 - Read data from bronze layer and overwrite data in stg table every time

df = spark.sql(
    """
    select
        file_name,
        file_surrogate_key,
        length,
        ai_parse_document(
            content,
            MAP('version', '2.0')
        ) AS parsed_content,
        src_inserted_stamp,
        operation_flag,
        hash_diff
    from (
        select
            file_name,
            md5(file_name) as file_surrogate_key,
            content,
            length,
            src_inserted_stamp,
            operation_flag,
            md5(
                concat_ws(
                    '||',
                    coalesce(trim(length), '<NULL>'),
                    coalesce(trim(parsed_content), '<NULL>')
                )
            ) as hash_diff,
            row_number() over (partition by file_name order by src_inserted_stamp desc) as rn
        from nleshin_catalog.bronze_layer.objects_description_journal
        -- where sys_inserted_stamp >= dateadd(MINUTE, -cast(:lookback_minutes as int), :date_interval_end) and sys_inserted_stamp < :date_interval_end
    )
    where rn = 1;
    """,
    args={
        # "date_interval_end": dbutils.widgets.get("date_interval_end"),
        # "lookback_minutes": dbutils.widgets.get("lookback_minutes")
    }
)

df.write.mode("overwrite").saveAsTable("nleshin_catalog.stg_layer.objects_description")